# Cleaning a PostgreSQL Database
![Clean PostgreSQL Database](Project_Image.jpeg)
("Project_Image.jpeg")

### The code in this project is meant for a SQL environment.
#### This project was done in September, 2025.

## 📖 Background
A superstore is interested in retrieving insights pertaining to day-to-day retail business questions such as what are the top performing products & the top categories based on profit margins, & how missing data can be imputed for quantity of products per order. To do so, the provided dataset will need to be cleaned & reformatted using SQL skills & concepts.



## 💾 The Data:
There are four tables available that pertain to orders, orders that are returned, sales members, & products. As seen in the table descriptions below, date fields have been written to the `orders` table as `TEXT`, whereas numeric fields like sales have been written to the `orders` table as `Double Precision`. Some of these types may need to be manipulated in some of the queries. 

### `orders`:
| Column | Definition | Data type | Comments |
|--------|------------|-----------|----------|
| `row_id`| Unique Record ID | `INTEGER` | - |
| `order_id` | Identifier for each order in table | `TEXT` | Connects to `order_id` in `returned_orders` table |
| `order_date` | Date when order was placed | `TEXT` | - |
| `market` | Market order_id belongs to | `TEXT` | - |
| `region` | Region Customer belongs to | `TEXT` | Connects to `region` in `people` table |
| `product_id` | Identifier of Product bought | `TEXT` | Connects to `product_id` in `products` table |
| `sales` | Total Sales Amount for the Line Item | `DOUBLE PRECISION` | - |
| `quantity` | Total Quantity for the Line Item | `DOUBLE PRECISION` | - |
| `discount` | Discount applied for the Line Item | `DOUBLE PRECISION` | - |
| `profit` | Total Profit earned on the Line Item | `DOUBLE PRECISION` | - |

### `returned_orders`:
| Column | Definition | Data type |
|--------|------------|-----------|
| `returned`| Yes values for Order / Line Item Returned | `TEXT` |
| `order_id` | Identifier for each order in table | `TEXT` |
| `market` | Market order_id belongs to | `TEXT` |

### `people`:
| Column | Definition | Data type |
|--------|------------|-----------|
| `person`| Name of Salesperson credited with Order | `TEXT` |
| `region` | Region Salesperson in operating in | `TEXT` |

### `products`:
| Column | Definition | Data type |
|--------|------------|-----------|
| `product_id`| Unique Identifier for the Product | `TEXT` |
| `category` | Category Product belongs to | `TEXT` |
| `sub_category` | Sub Category Product belongs to | `TEXT` |
| `product_name` | Detailed Name of the Product | `TEXT` |

# Objectives
1) Find the top five products from each category based on highest total sales. The output should be sorted by `category` in ascending order & by `sales` in descending order within each category; i.e. within each category, the product with highest margin should sit on the top. The output should contain the following columns:
- `category`
- `product_name`
- `product_total_sales` (rounded to 2 decimal places)
- `product_total_profit` (rounded to 2 decimal places)
- `product_rank`


2) Calculate the quantity for orders with missing values in the `quantity` column by determining the unit price for each `product_id` using available order data, considering relevant pricing factors such as discount, market, or region. Then, use this unit price to estimate the missing quantity values. The calculated values should be stored in the `calculated_quantity` column. The output should contain the following columns:
- `product_id`
- `discount`
- `market`
- `region`
- `sales`
- `quantity`
- `calculated_quantity` (rounded to 0 decimal places)

# Summary of Findings
This project utilized arithmetic & SQL tools to evaluate the most popular products sold & identify & impute orders with missing data points. More specifically, common table expressions (CTEs), window functions, type casting, joins, & subqueries were utilized to manipulate the provided data. The two sections of analyses primarily dealt with the `orders` table, but the `products` table was also utilized. The former contained 51,290 data points.

### Analysis I - Top Products by Sales
The first section examined the top-selling products across each of the three categories according to total sales. By manipulating data types & utilizing a subquery, window function, & left join, these items were obtained & sorted according to their category & total sales.  
Generally, technology items yielded the most in terms of sales & profits, whereas office-supplies items generated the least.
- Of the fifteen items obtained in this exercise, the item that generated the most in terms of sales was the "Apple Smart Phone, Full Size" (~ 86,935); however, these orders only generated the ninth most profits (~ 5,921).
- The item that generated the most in terms of profits was the "Canon imageCLASS 2200 Advanced Copier" (~ 25,199); however, these orders only generated the fifth most sales (~ 61,599).

These items in particular are some of the most important for the superstore to continue to restock because of how much they have contributed to their total sales. Additional analyses could be done to evaluate how often such products are purchased which can give an idea as to how often the superstore will need to buy & stock such items.


### Analysis II - Imputing Order Quantities
Lastly, there were five data points in the `orders` table that possessed missing quantity values. To impute them, some particular arithmetic & SQL tools were utilized.  
Firstly, in order to obtain the quantity of items bought in an order, the unit price of an item needed to be obtained. Given that this variable was not present in the provided data, it needed to be built. To obtain the unit price, the following formula was used: (sales / (1-discount)) / (quantity); however, for orders with a missing quantity, this would yield a missing unit price too. As such, a unit-price imputation value needed to be obtained.

To build these imputation values, other variables were considered. More specifically, of the five data points with a missing quantity, each corresponded to a unique product id. Each of these product id's accounted for between 5-10 orders. The non-missing sales, discount, & quantity values for each of these five product id's were used to calculate an associated unit price for each order. This data was assembled into a CTE. Next, a window function was used in second CTE to obtain the average unit price for each product id using these new unit prices. These averages then acted as the basis for the imputed quantity values, the calculations of which used the following formula: ( (sales / (1-discount)) / (imp_unit_price) ).

On another note, there don't seem to be any particular commonalites between these data points with missing 'quantity' values. Additionally, given that only five of the 51,290 data points in the `orders` table have a missing 'quantity', it is hard to point to some a particular issue or common denominator regarding the root of these missing values. Such a small number of cases is more indicative of less reproducible problems, like entry errors, as opposed to larger underlying problems (e.g. software issue, data tracking issue). Nevertheless, these data points should be investigated to determine the full scope of the problem & what solutions could be devised to prevent them from occurring in the future.

------------------------------------------------------------------------------------------------------------------------------

# Analysis I - Top Products by Sales
Find the top five products from each category based on highest total sales. The output should be sorted by `category` in ascending order & by `sales` in descending order within each category; i.e. within each category, the product with highest margin should sit on the top. The output should contain the following columns:
- `category`
- `product_name`
- `product_total_sales` (rounded to 2 decimal places)
- `product_total_profit` (rounded to 2 decimal places)
- `product_rank`

**Plan of actions:**
- In the `orders` table, there are three unique categories: Furniture, Office Supplies, Technology. Since the data is being limited to the five most popular products in terms of sales, the query should produce no more than 15 rows.
- In order to specify the decimal precision of numerical values in PostgreSQL, the DOUBLE PRECISION types need to be converted to NUMERIC types.
- To retrieve both products' sales & categories, the `orders` & `products` tables need to be joined, which can be done using the 'product_id' variable. Since this inquiry concerns order details (e.g. total sales), a left join can be used to retrieve all orders. It's possible that not all products have been purchased in orders, so joining all products may not be necessary.
- To obtain total sales & profit, the 'sales' & 'profit' variables can be used respectively.
- To obtain the top five products according to sales, ranks can be applied according to total sales by category. This would necessitate using a window function which would further require a CTE/subquery. In this case, a subquery was used in the FROM clause. The data can then be filtered for ranks of five or less.
- Make sure that the query is organized by category (ascending) & total sales (descending).

In [ ]:
SELECT category, product_name, product_total_sales, product_total_profit, product_rank
FROM (
    SELECT product_name, category, 
        ROUND(SUM(sales)::NUMERIC, 2) AS product_total_sales, 
        ROUND(SUM(profit)::NUMERIC, 2) AS product_total_profit, 
        ROW_NUMBER() OVER(PARTITION BY category ORDER BY SUM(sales) DESC) AS product_rank
    FROM orders
    LEFT JOIN products
    USING(product_id)
    GROUP BY product_name, category
    ORDER BY category, product_total_sales DESC) AS subq
WHERE product_rank < 6;
    #produces 15 rows

#the output of this query is shown directly below

![Q1_output.png](Q1_output.png)
("Q1_output.png")

This query yields the five-most popular products across the three product categories in terms of total sales generated. It contains 15 rows.  
Based on these items, technology products were clearly the most valuable items that were sold, whereas office-supplies items were the least valuable among the three product categories.

On another note, total profits are generally correlated with total sales _across_ the product categories, at least based on these top-five subsets of items. More specifically, total profits were higher across the most sold technology items (their profits ranged between 5,900-25,200), whereas total profits were lowest for office supplies items (their profits ranged between -2,180-11,650) which saw the least sales.
- Of these 15 products, one of them had a negative profit margin: "Hoover Stove, White" had a total profit of about -2,180. This is a very strange data point because the third most valuable item in the office-supplies category was "Hoover Stove, Red," which had a total profit of about 11,650.

Of the **furniture** items that made the most in terms of sales:
- It appears that the company was quite proficient in selling Executive Leather Armchairs since the top five products, in terms of sales, were such an item.
- The "Harbour Creations Executive Leather Armchair..." yielded the third most in terms of sales (50,121), but they were the most profitable item (10,427).
- The "Office Star Executive Leather Armchair..." yielded the second most in terms of sales (51,449), but they were the least profitable item (4,925).
- These five items each totaled more than 40,000 in sales each & up to 59,000. In terms of profit, they yielded between 4,900 - 10,500.

Of the **office supplies** items that made the most in terms of sales:
- It appears that the company sold hoover stoves & file carts quite proficiently since they account for four of the top five products according to total sales.
- The "Hoover Stove, Red" yielded the third most in terms of sales (32,644), but they were the most profitable item (11,651).
- The "Hoover Stove, White" yielded the second most in terms of sales (32,842), but they were the least profitable item (-2,180). In fact, they failed to yield a positive profit. For whatever reason, white Hoover stoves yielded nearly 14,000 less in profit than red Hoover stoves.
    - The discrepancy between the profits of these two items, despite them only differing in color, would be worth investigating further. Their total sales are very similar so perhaps this inconsistency points to a considerable difference in price & number of units sold; however, why would these two products differ so significantly in price if they only differ in color?
- These five items totaled more than 28,000 in sales each & up to 40,000. In terms of profit, they yielded between -2,200 - 12,000.

Of the **technology** items that made the most in terms of sales:
- It appears that the company was quite proficient in selling smart phones since they account for four of the top five products according to total sales.
- The "Canon imageCLASS 2200 Advanced Copier" yielded the fifth most in terms of sales (61,599), but they were the most profitable item (25,199).
- The "Apple Smart Phone, Full Size" yielded the most in terms of sales (86,935), but they were the least profitable item (5,921).
- These five item totaled more than 61,000 in sales each & up to 87,000. In terms of profit, they yielded between 5,900 - 26,000.

# Analysis II - Imputing Order Quantities
Calculate the quantity for orders with missing values in the `quantity` column by determining the unit price for each `product_id` using available order data, considering relevant pricing factors such as discount, market, or region. Then, use this unit price to estimate the missing quantity values. The calculated values should be stored in the `calculated_quantity` column. The output should contain the following columns:
- `product_id`
- `discount`
- `market`
- `region`
- `sales`
- `quantity`
- `calculated_quantity` (rounded to 0 decimal places)


**Plan of action:**
Upon the investigating the `orders` table, which has 51,290 data points & 10,292 unique 'product_id's, there are five rows with missing 'quantity' values (& five unique orders).
- The product id's with missing quantities are: "TEC-STA-10003330" (eight unique orders), "FUR-ADV-10000571" (eight unique orders), "FUR-BO-10001337" (ten unique orders), "TEC-STA-10004542" (five unique orders), "FUR-ADV-10004395" (ten unique orders). Examining the non-missing 'quantity' values (& other variables) in orders with these product id's could help provide a sense of what the missing values might/should be.

In order to calculate missing 'quantity' values, the unit price of a product needs to be known which is not in any of the tables provided. Therefore, for orders with missing quantities, this variable cannot be obtained. Alternatively, another way to calculate a product's unit price could involve calculating the average unit price across all orders for each product. To do so, a window function can be used. These average unit prices could then be used to impute the missing quantities of orders specifically with missing 'quantity' values.

To calculate these missing 'quantity' values, some mathematical equations can be considered. Generally speaking, total sales of an order can be written as: quantity * unit_price. When discounts are applied, this formula becomes: (quantity * unit_price) * (1-discount). To determine the unit price of a product, this formula becomes: unit_price = (sales / (1-discount)) / (quantity).
- As an example, if an order buys 10 phone's, each at a price of \\$200 & discounted at 10%, then the total sales will be: \\$1,800.

With this in mind, this problem will require some nuance given the variables that are desired in the final output. Firstly, the `orders` table needs to be truncated to all orders whose 'product_id's are missing in order to retrieve all data points with these specific 'product_id's. These product id's, which can be retrieved with a subquery in the WHERE clause, account for 41 orders.  
Using these 41 data points, the unit prices can be calculated using the formula described above. In order to apply a window function to this unit-price quantity, CTEs (common table expressions) can be built. So, one CTE will contain the 41 data points & another can calculate & append the average unit prices for each product id with a missing 'quantity'.  
These average unit prices can then be used to calculate the missing 'quantity' values using the formula: calculated_quantity = (sales / (1-discount)) / (imputed_unit_price). To round these values to zero decimals, they can be converted to INTEGER types.

In [ ]:
#Make a CTE to retrieve the desired variables & calculate the unit prices for 'product_id's accounting for missing 'quantity's.
    #These unit prices are based on 'sales', 'discount', & 'quantity'.
    #The 5 corresponding 'product_id's with missing 'quantity's account for 41 orders in total.
    #Data points with a missing 'quantity' will have a missing unit price.
WITH cte1 AS (
    SELECT row_id, product_id, market, region, sales, discount, quantity,
        ( (sales / (1-discount)) / quantity) AS calc_unit_price,
        quantity AS calc_quantity
    FROM orders
    WHERE product_id IN (
        SELECT product_id
        FROM orders
        WHERE quantity IS NULL)
    ORDER BY product_id),

#Using the CTE above, which generates 41 rows, can obtain imputation values for the unit prices of 'product_id's with missing
    #'quantity's. These unit-price imputation values will be calculated using the average unit prices for each 'product_id'.
    #A 2nd CTE was made so that the computed metric ("imp_unit_price") could be referenced directly later rather than 
    #having to rewrite the whole line of code again. In the window function, can use AVG because the non-missing unit prices 
    #for each "product_id" are identical.
cte2 AS (
    SELECT *, AVG(calc_unit_price) OVER(PARTITION BY product_id) AS imp_unit_price
    FROM cte1 ),

#Now that the "imp_unit_price" variable has been made, use it to impute for 'quantity'.
cte3 AS (
    SELECT *, ( (sales / (1-discount) ) / (imp_unit_price))::INT AS imp_quantity
    FROM cte2 )
#Check that the imputed quantity = actual quantity --> GOOD
    #WHERE quantity != ( (sales / (1-discount) ) / (imp_unit_price))::INT;

#Produce the desired query
SELECT product_id, discount, market, region, sales, quantity, imp_quantity AS calculated_quantity
FROM cte3
WHERE quantity IS NULL;
    #produces 5 rows; the output of this query is shown directly below


#Can reintegrate the imputed data into the main dataset. For data points that didn't go through this process above, just use 
    #the actual 'quantity' variable. Recall that there are only 5 data points w/missing 'quantity'. For the other 51,285 data 
    #points without an imputed 'quantity' value, just use the actual 'quantity' value.
#SELECT O.product_id, O.discount, O.market, O.region, O.sales, O.quantity, 
    #COALESCE(cte3.imp_quantity, O.quantity) AS calculated_quantity
#FROM orders AS O
#FULL JOIN cte3
#USING(row_id);

![Q2_output.png](Q2_output.png)
("Q2_output.png")

The image above lists the five data points with a missing 'quantity' along with the expected quantity based on the average unit prices of orders with non-missing 'quantity's. These averages are calculated according to the 'product_id' with a missing order 'quantity'.

To integrate these calculated 'quantity' values into the original dataset, multiple methods can be made. The UPDATE command can be used which would follow some syntax along the lines of:
- UPDATE orders       SET quantity = (SELECT COALESCE(quantity, calculated_quantity) FROM cte3)        WHERE quantity IS NULL.

Alternatively, a join could be used to produce a new table with the complete data such as:
- WITH full_orders AS (SELECT *, COALESCE(quantity, imp_quantity)      FROM cte3)

On another note, there don't seem to be any particular commonalites between these data points with missing 'quantity' values. Additionally, given that only five of the 51,290 data points in the `orders` table have a missing 'quantity', it is hard to point to some a particular issue or common denominator regarding the root of these missing values. Such a small number of cases is more indicative of less reproducible problems, like entry errors, as opposed to larger underlying problems (e.g. software issue, data tracking issue).